In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-40s :: %(message)s'
)


## Inverse obstacle problem with Dirichlet boundary condition

We consider the operator that maps the shape of a sound-soft obstacle $D$ to the far-field measurements. 
The scattering problem is described by

$$
        \begin{cases}
            \Delta u +\kappa^2 u = 0 & \text{ in } \mathbb{R}^2\backslash\overline{D}\\
             u = 0  & \text{ on } \partial D\\
            \displaystyle{\lim_{r\to\infty}}r^{\frac{1}{2}}(\frac{\partial u^s}{\partial r}-i\kappa u^s)=0 & \text{ for } r=|x|,
        \end{cases}
$$
where $u=u^s+u^i$ is the total field generated by a plane incident wave $u^i(x)=\exp(i\kappa x\cdot d)$ in direction 
$d\in S^1$, 
and $D$ is a bounded obstacle in $\mathbb{R}^2$ with $\partial D\in\mathcal{C}^2$.
The far field pattern $u^{\infty}$ of the scattered field $u^s$ is defined by the asymptotic relation
$$
u^s(x) = \frac{e^{i\kappa|x|}}{\sqrt{|x}} u^{\infty}\left(\frac{x}{|x|}\right)\left(1+O\left(\frac{1}{|x|}\right)\right),\qquad 
|x|\to \infty.
$$ 
The forward operator maps a parameterization of $\partial D$ to a complex matrix, the columns of which are far field patterns 
corresponding to incident waves from different directions.      

In [ ]:
from  dirichlet_op import DirichletOp
op = DirichletOp(
    kappa = 4,
    N_inc=4,
    N_meas=64,
    N_ieq=64
)

### Neumann boundary condition
Instead of the Dirichlet condition $u=0$ on $\partial D$, we can also consider a Neumann condition for the total field, 
corresponding to sound-hard obstacles:
$$
\frac{\partial u}{\partial \nu}=0\qquad \text{on }\partial D. 
$$

In [ ]:
"""from neumann_op import NeumannOp
op = NeumannOp(
    kappa = 4,
    inc_waves=4,
    meas_dir=64,
    N_ieq=64
)"""

### Transmission conditions 
Scattering by a homogeneous penetrable obstacle can be described by transmission conditions: 
$$
        \begin{cases}
            \Delta u^{int} +\kappa_i^2 u^{int} = 0 & \text{ in } D \\
            \Delta u^{s} +\kappa_e^2 u^{s} = 0 & \text{ in } \mathbb{R}^2\backslash\overline{D}\\
             u^{int}=u & \text{ on } \partial D\\
            \frac{\partial u^{int}}{\partial\nu}=\rho\frac{\partial u}{\partial\nu} & \text{ on }\partial D\\
            \displaystyle{\lim_{r\to\infty}}r^{\frac{1}{2}}(\frac{\partial u^s}{\partial r}-i\kappa u^s)=0 &\text{ for } r=|x|.
        \end{cases}
$$
Here $\rho\in\mathbb{C}\backslash 0$, and $u=u^{s}+u^{i}$ is the total field in $\mathbb{R}^2\backslash\overline{D}$.


In [ ]:
"""from transmission_op import TransmissionOp
op = TransmissionOp(
    kappa_in = 4,
    kappa_ex = 8,
    N_inc=4,
    rho = 2., 
    N_meas=64,
    N_ieq=64
)"""


import some curve as ground truth and create corresponding synthetic data

In [ ]:
from regpy.vecsps.curve import apple
farfield, exact_solution = op.create_synthetic_data(apple,N_ieq_synth=196)


plot the exact synthetic data

In [ ]:
x= op.codomain.coords[0]
fig, axs = plt.subplots(1, 2,figsize=(10,5))
number2show = 4
for j in range(number2show):
    y = farfield[:,j]
    axs[0].plot(x,np.abs(y))
    axs[1].plot(x,np.unwrap(np.angle(y)))
fig.suptitle(f'farfield patterns for first {number2show} of {farfield.shape[1]} incident waves')
axs[0].set_title('absolute values')
axs[1].set_title('phases')
axs[0].set_xlabel('Measurement direction (rad)')
_= axs[1].set_xlabel('Measurement direction (rad)')


Get initial guess, add noise to data, and create setting

In [ ]:
from regpy.solvers import Setting
from regpy.hilbert import L2, Sobolev

#Initial guess
t = 2*np.pi*np.arange(0, op.N_FK)/op.N_FK
init = 0.45*np.vstack((np.cos(t), np.sin(t))).T

setting = Setting(op=op, penalty=Sobolev, data_fid=L2)

# add Gaussian noise to exact data
noise_level = 0.01
noise = op.codomain.randn()
noise = noise_level*setting.h_codomain.norm(farfield)/setting.h_codomain.norm(noise)*noise
data = farfield+noise


perform the inversion

In [ ]:
from regpy.solvers.nonlinear.irgnm import IrgnmCG
from regpy.solvers.nonlinear.newton import NewtonCG
import regpy.stoprules as rules

#Solver: NewtonCG or IrgnmCG

solver = NewtonCG(
    setting, 
    data, 
    init = init,
    cgmaxit=50, 
    rho=0.6
)
"""

solver = IrgnmCG(
    setting, data,
    regpar=1.,
    regpar_step=0.5,
    init=init
)"""

stoprule = (
    rules.CountIterations(25) +
    rules.Discrepancy(
        setting.h_codomain.norm, data,
        noiselevel=noise_level,
        tau=2.1
    )
)
reco, reco_data = solver.run(stoprule)

Plot results

In [ ]:
fig, axs = plt.subplots(1, 2,figsize=(10,5))
axs[0].set_title('Obstacle')
axs[1].set_title('Farfield (real part,first inc. wave)')

reco_curve = op.domain.coeff2curve(reco)
axs[0].plot(*exact_solution.z,label='exact')
axs[0].plot(*reco_curve.z,label='reco')
axs[0].legend()

axs[1].plot(op.codomain.coords[0][:,0], farfield.imag[:,0], label='exact')
axs[1].plot(op.codomain.coords[0][:,0], reco_data.imag[:,0], label='reco')
axs[1].plot(op.codomain.coords[0][:,0], data.imag[:,0], '.', label='measured')
axs[1].legend()
plt.show()